# Verificando estrutura dos arquivos de treino

In [ ]:
import tensorflow as tf
import glob

BASE_PATH = '/content/drive/MyDrive/IA-Cristalizacao'
files = glob.glob(BASE_PATH + '/data/train-*')

if files:
    raw_dataset = tf.data.TFRecordDataset(files[0])
    for raw_record in raw_dataset.take(1):
        example = tf.train.Example()
        example.ParseFromString(raw_record.numpy())
        print("=== ESTRUTURA REAL DO SEU TFRECORD ===")
        print(example.features.feature.keys())
else:
    print("Nenhum arquivo encontrado em:", BASE_PATH + '/data/')

=== ESTRUTURA REAL DO SEU TFRECORD ===
KeysView({'image/width': int64_list {
  value: 1280
}
, 'image/class/source': int64_list {
  value: 3
}
, 'image/colorspace': bytes_list {
  value: "RGB"
}
, 'image/class/text': bytes_list {
  value: "Clear"
}
, 'image/height': int64_list {
  value: 960
}
, 'image/class/label': int64_list {
  value: 0
}
, 'image/encoded': bytes_list {
  value: "\377\330\377\340\000\020JFIF\000\001\001\001\000`\000`\000\000\377\333\000C\000\007\005\005\006\005\004\007\006\005\006\010\007\007\010\n\021\013\n\t\t\n\025\017\020\014\021\030\025\032\031\030\025\030\027\033\036'!\033\035%\035\027\030"."%()+,+\032 /3/*2'*+*\377\333\000C\001\007\010\010\n\t\n\024\013\013\024*\034\030\034**************************************************\377\300\000\021\010\003\300\005\000\003\001"\000\002\021\001\003\021\001\377\304\000\037\000\000\001\005\001\001\001\001\001\001\000\000\000\000\000\000\000\000\001\002\003\004\005\006\007\010\t\n\013\377\304\000\265\020\000\002\001\003\003

# Treino 1

In [ ]:
import tensorflow as tf
import glob
import os

BASE_PATH = '/content/drive/MyDrive/IA-Cristalizacao'
SAVE_PATH = BASE_PATH + '/marco_model.keras'

TRAIN_FILES = glob.glob(BASE_PATH + '/data/train-*')
VAL_FILES = glob.glob(BASE_PATH + '/data/test-*')

BATCH_SIZE = 16
EPOCHS = 10

if not os.path.exists('/content/drive'):
    from google.colab import drive
    print("Ligando ao Google Drive...")
    drive.mount('/content/drive')

print(f"Arquivos de treino encontrados: {len(TRAIN_FILES)}")
print(f"Arquivos de validação encontrados: {len(VAL_FILES)}")

def parse_tfrecord_fn(example):

    feature_description = {
        'image/encoded': tf.io.FixedLenFeature([], tf.string),
        'image/class/label': tf.io.FixedLenFeature([], tf.int64),
    }
    example = tf.io.parse_single_example(example, feature_description)

    image = tf.io.decode_jpeg(example['image/encoded'], channels=3)
    image = tf.image.resize(image, [224, 224])
    image = tf.cast(image, tf.float32) / 255.0

    return image, example['image/class/label']

def load_dataset(filenames, is_training=True):
    dataset = tf.data.TFRecordDataset(filenames, num_parallel_reads=tf.data.AUTOTUNE)
    dataset = dataset.map(parse_tfrecord_fn, num_parallel_calls=tf.data.AUTOTUNE)


    if is_training:
        dataset = dataset.shuffle(1000)

    dataset = dataset.batch(BATCH_SIZE, drop_remainder=True).repeat().prefetch(tf.data.AUTOTUNE)
    return dataset

print("Carregando datasets de alta performance...")
train_ds = load_dataset(TRAIN_FILES, is_training=True)
val_ds = load_dataset(VAL_FILES, is_training=False)

if os.path.exists(SAVE_PATH):
    print("\n[INFO] Save state encontrado no Drive! Carregando o progresso para continuar...")
    model = tf.keras.models.load_model(SAVE_PATH)
else:
    print("\n[INFO] Nenhum save encontrado. Iniciando o treinamento do total zero...")

    base_model = tf.keras.applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = True

    for layer in base_model.layers[:100]:
        layer.trainable = False

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(4, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )


callbacks_list = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(filepath=SAVE_PATH, monitor='val_loss', save_best_only=True)
]

TRAIN_STEPS = (len(TRAIN_FILES) * 1024) // BATCH_SIZE
VAL_STEPS = (len(VAL_FILES) * 1024) // BATCH_SIZE

print(f"Passos de treino por época: {TRAIN_STEPS}")
print(f"Passos de validação por época: {VAL_STEPS}")
print("\nIniciando execução segura na GPU T4 remota...")

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=TRAIN_STEPS,
    validation_steps=VAL_STEPS,
    callbacks=callbacks_list
)

print(f"\n[SUCESSO] Treinamento concluído! Modelo final salvo em: {SAVE_PATH}")

Arquivos de treino encontrados: 100
Arquivos de validação encontrados: 46
Carregando datasets de alta performance...

[INFO] Save state encontrado no Drive! Carregando o progresso para continuar...
Passos de treino por época: 6400
Passos de validação por época: 2944

Iniciando execução segura na GPU T4 remota...
Epoch 1/10
6400/6400 ━━━━━━━━━━━━━━━━━━━━ 1129s 171ms/step - accuracy: 0.7789 - loss: 0.5893 - val_accuracy: 0.7828 - val_loss: 0.5777
Epoch 2/10
6400/6400 ━━━━━━━━━━━━━━━━━━━━ 1092s 171ms/step - accuracy: 0.8026 - loss: 0.5308 - val_accuracy: 0.7642 - val_loss: 0.7245
Epoch 3/10
6400/6400 ━━━━━━━━━━━━━━━━━━━━ 1096s 171ms/step - accuracy: 0.8195 - loss: 0.4890 - val_accuracy: 0.7593 - val_loss: 0.6099
Epoch 4/10
6400/6400 ━━━━━━━━━━━━━━━━━━━━ 1111s 174ms/step - accuracy: 0.8322 - loss: 0.4521 - val_accuracy: 0.8161 - val_loss: 0.4972
Epoch 5/10
6400/6400 ━━━━━━━━━━━━━━━━━━━━ 1097s 171ms/step - accuracy: 0.8468 - loss: 0.4155 - val_accuracy: 0.7745 - val_loss: 0.5759
Epoch 6/10
